In [1]:
# Purpose: Load and inspect the engineered Sahiwal soil dataset for agricultural intelligence.

import pandas as pd
import numpy as np

data_path = "../data/processed/sahiwal_soil_dataset_engineered.csv"

soil_data = pd.read_csv(data_path)

print("Dataset shape:", soil_data.shape)
print("\nAvailable properties:")
print(soil_data.columns.tolist())

print("\nDataset information:")
soil_data.info()

print("\nStatistical summary:")
display(soil_data.describe().round(3))

Dataset shape: (48807, 10)

Available properties:
['longitude', 'latitude', 'ph', 'soc_g_kg', 'nitrogen_g_kg', 'clay_pct', 'sand_pct', 'silt_pct', 'cec_cmol_kg', 'soil_texture']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 48807 entries, 0 to 48806
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   longitude      48807 non-null  float64
 1   latitude       48807 non-null  float64
 2   ph             48807 non-null  float64
 3   soc_g_kg       48807 non-null  float64
 4   nitrogen_g_kg  48807 non-null  float64
 5   clay_pct       48807 non-null  float64
 6   sand_pct       48807 non-null  float64
 7   silt_pct       48807 non-null  float64
 8   cec_cmol_kg    48807 non-null  float64
 9   soil_texture   48807 non-null  str    
dtypes: float64(9), str(1)
memory usage: 3.7 MB

Statistical summary:


,longitude,latitude,ph,soc_g_kg,nitrogen_g_kg,clay_pct,sand_pct,silt_pct,cec_cmol_kg
count,48807.000,48807.000,48807.000,48807.000,48807.000,48807.000,48807.000,48807.000,48807.000
mean,72.880,30.555,7.799,10.451,1.245,22.522,42.953,34.525,15.264
std,0.246,0.146,0.098,0.876,0.271,1.429,2.014,2.263,1.162
min,72.406,30.201,7.400,6.400,0.780,18.300,36.200,25.100,11.400
25%,72.668,30.451,7.800,9.900,1.080,21.500,41.600,33.000,14.300
50%,72.888,30.558,7.800,10.400,1.150,22.400,42.800,34.600,15.200
75%,73.090,30.664,7.900,11.000,1.280,23.400,44.100,36.100,16.100
max,73.342,30.882,8.200,23.600,2.350,31.100,50.900,41.700,19.400


In [2]:
# Purpose: Examine soil-property ranges, texture composition, and potential data-quality issues.

soil_properties = [
    'ph', 'soc_g_kg', 'nitrogen_g_kg',
    'clay_pct', 'sand_pct', 'silt_pct', 'cec_cmol_kg'
]

print("SOIL PROPERTY RANGES")
display(soil_data[soil_properties].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
).T.round(3))

print("\nSOIL TEXTURE DISTRIBUTION")
display(
    soil_data['soil_texture']
    .value_counts()
    .to_frame('Sample Count')
    .assign(Percentage=lambda x: x['Sample Count'] / len(soil_data) * 100)
    .round(2)
)

print("\nTEXTURE COMPONENT SUM")
texture_sum = (
    soil_data['clay_pct']
    + soil_data['sand_pct']
    + soil_data['silt_pct']
)

print("Mean:", round(texture_sum.mean(), 3))
print("Minimum:", round(texture_sum.min(), 3))
print("Maximum:", round(texture_sum.max(), 3))

SOIL PROPERTY RANGES


,count,mean,std,min,5%,25%,50%,75%,95%,max
ph,48807.0,7.799,0.098,7.40,7.60,7.80,7.80,7.90,7.90,8.20
soc_g_kg,48807.0,10.451,0.876,6.40,9.10,9.90,10.40,11.00,11.80,23.60
nitrogen_g_kg,48807.0,1.245,0.271,0.78,0.99,1.08,1.15,1.28,1.93,2.35
clay_pct,48807.0,22.522,1.429,18.30,20.40,21.50,22.40,23.40,24.90,31.10
sand_pct,48807.0,42.953,2.014,36.20,39.90,41.60,42.80,44.10,46.50,50.90
silt_pct,48807.0,34.525,2.263,25.10,30.80,33.00,34.60,36.10,38.20,41.70
cec_cmol_kg,48807.0,15.264,1.162,11.40,13.60,14.30,15.20,16.10,17.20,19.40



SOIL TEXTURE DISTRIBUTION


,Sample Count,Percentage
soil_texture,,
loam,48203,98.76
clay loam,345,0.71
sandy clay loam,259,0.53



TEXTURE COMPONENT SUM
Mean: 100.0
Minimum: 99.9
Maximum: 100.1


In [3]:
# Purpose: Measure spatial variability in soil properties across Sahiwal.

spatial_variability = []

for feature in soil_properties:
    values = soil_data[feature]

    spatial_variability.append({
        'Property': feature,
        'Mean': values.mean(),
        'Standard Deviation': values.std(),
        'Coefficient of Variation (%)': (
            values.std() / values.mean()
        ) * 100,
        'Range': values.max() - values.min()
    })

variability_df = pd.DataFrame(spatial_variability)

display(variability_df.round(3))

,Property,Mean,Standard Deviation,Coefficient of Variation (%),Range
0,ph,7.799,0.098,1.252,0.80
1,soc_g_kg,10.451,0.876,8.378,17.20
2,nitrogen_g_kg,1.245,0.271,21.782,1.57
3,clay_pct,22.522,1.429,6.343,12.80
4,sand_pct,42.953,2.014,4.689,14.70
5,silt_pct,34.525,2.263,6.555,16.60
6,cec_cmol_kg,15.264,1.162,7.610,8.00


### Spatial Variability Assessment

The coefficient of variation (CV) was calculated to compare the relative variability of soil properties across Sahiwal.

Key findings:

- Nitrogen showed the highest relative variability (21.78%), making it an important property for further spatial investigation.
- Soil organic carbon (SOC) showed 8.38% variability.
- CEC showed 7.61% variability.
- Sand, clay, and silt showed relatively lower variability.
- Soil pH showed the lowest variability (1.25%), indicating relatively consistent estimated pH across the district.

These results describe district-wide variation, not the geographical distribution of high- and low-value areas. CV values also do not establish whether a soil property is agronomically sufficient or deficient.

**Limitation:** All measurements are SoilGrids estimates for the 0–5 cm soil layer, not laboratory measurements or direct assessments of plant nutrient availability.